# Fase 1.2 — Extracción, Transformación y Carga (ETL) del dataset Olist

**Proyecto:** Desarrollo de un Proyecto de Análisis de Datos y Modelo Predictivo para una Aplicación de Negocio
**Materia:** Análisis de Datos — Ingeniería de Sistemas
**Fecha:** Mayo de 2026
**Dataset seleccionado en la fase 1.1:** *Brazilian E-Commerce Public Dataset by Olist* (9 archivos CSV, ~100k órdenes, 2016-2018).


## Contenido

1. Diseño del modelo de datos (esquema estrella)
2. Configuración y utilidades
3. **Extract** — lectura tipada de los 9 CSV
4. **Profile** — perfilado de calidad inicial
5. **Transform** — limpieza, normalización y construcción de dimensiones/hechos
6. **Validate** — validaciones de integridad y reglas de negocio
7. **Load** — persistencia de tablas limpias (CSV)
8. **Tabla analítica** — denormalización por orden para EDA/BI/modelado
9. Reporte ejecutivo del ETL
10. Próximos pasos


## 1. Diseño del modelo de datos

A partir de los 9 archivos crudos de Olist se construye un **modelo
dimensional con esquema estrella**: una tabla de hechos central rodeada
de dimensiones descriptivas. Este modelo facilita las fases siguientes
(EDA, BI y modelado).

### Estructura de carpetas resultante

```
Data/
 ├── olist_*.csv                       (archivos originales sin tocar)
 ├── product_category_name_translation.csv
 └── processed/                        (salidas de este notebook)
      ├── dim_customers.csv
      ├── dim_sellers.csv
      ├── dim_products.csv
      ├── dim_geography.csv
      ├── dim_date.csv
      ├── dim_reviews.csv
      ├── fact_order_items.csv
      ├── fact_payments.csv
      └── marts/
           └── mart_orders.csv         (tabla denormalizada por orden)
```

### Esquema estrella

```
                 ┌──────────────────┐
                 │    dim_date      │
                 └────────┬─────────┘
                          │
┌──────────────┐   ┌──────┴─────────┐   ┌───────────────┐
│ dim_customers├──>│  fact_order_   │<──┤  dim_sellers  │
└──────────────┘   │     items      │   └───────────────┘
                   │                │
┌──────────────┐   │                │   ┌───────────────┐
│ dim_products ├──>│                │<──┤ dim_geography │
└──────────────┘   └──────┬─────────┘   └───────────────┘
                          │
                 ┌────────┴─────────┐
                 │  fact_payments   │
                 └──────────────────┘
```

- **Grano de la tabla de hechos `fact_order_items`:** *un ítem dentro de
  una orden* (`order_id` + `order_item_id`). Permite agregaciones a
  nivel ítem, orden o cliente.
- **Grano de `fact_payments`:** *un pago dentro de una orden*
  (`order_id` + `payment_sequential`). Una orden puede tener pagos
  fraccionados.
- Las dimensiones se actualizan por sobrescritura (no se conserva
  historial), pues el dataset es cerrado.


## 2. Configuración y utilidades

Centralizamos rutas, tipados y funciones auxiliares para hacer el
proceso reproducible y fácil de mantener. Cada función está
documentada con docstring para soportar la trazabilidad técnica.


In [1]:
from __future__ import annotations

import time
import unicodedata
from pathlib import Path
from typing import Callable

import numpy as np
import pandas as pd

# --- Rutas ---------------------------------------------------------
NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR  = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "Notebooks" else NOTEBOOK_DIR
RAW_DIR       = PROJECT_DIR / "Data"
PROCESSED_DIR = PROJECT_DIR / "Data" / "processed"
MART_DIR      = PROCESSED_DIR / "marts"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MART_DIR.mkdir(parents=True, exist_ok=True)

# --- Diccionario de tipados controlados por archivo ----------------
SCHEMAS: dict[str, dict] = {
    "olist_customers_dataset.csv": {
        "dtype": {
            "customer_id": "string",
            "customer_unique_id": "string",
            "customer_zip_code_prefix": "string",
            "customer_city": "string",
            "customer_state": "string",
        },
    },
    "olist_geolocation_dataset.csv": {
        "dtype": {
            "geolocation_zip_code_prefix": "string",
            "geolocation_lat": "float64",
            "geolocation_lng": "float64",
            "geolocation_city": "string",
            "geolocation_state": "string",
        },
    },
    "olist_orders_dataset.csv": {
        "dtype": {
            "order_id": "string",
            "customer_id": "string",
            "order_status": "string",
        },
        "parse_dates": [
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
        ],
    },
    "olist_order_items_dataset.csv": {
        "dtype": {
            "order_id": "string",
            "order_item_id": "Int64",
            "product_id": "string",
            "seller_id": "string",
            "price": "float64",
            "freight_value": "float64",
        },
        "parse_dates": ["shipping_limit_date"],
    },
    "olist_order_payments_dataset.csv": {
        "dtype": {
            "order_id": "string",
            "payment_sequential": "Int64",
            "payment_type": "string",
            "payment_installments": "Int64",
            "payment_value": "float64",
        },
    },
    "olist_order_reviews_dataset.csv": {
        "dtype": {
            "review_id": "string",
            "order_id": "string",
            "review_score": "Int64",
            "review_comment_title": "string",
            "review_comment_message": "string",
        },
        "parse_dates": ["review_creation_date", "review_answer_timestamp"],
    },
    "olist_products_dataset.csv": {
        "dtype": {
            "product_id": "string",
            "product_category_name": "string",
            "product_name_lenght": "Int64",
            "product_description_lenght": "Int64",
            "product_photos_qty": "Int64",
            "product_weight_g": "float64",
            "product_length_cm": "float64",
            "product_height_cm": "float64",
            "product_width_cm": "float64",
        },
    },
    "olist_sellers_dataset.csv": {
        "dtype": {
            "seller_id": "string",
            "seller_zip_code_prefix": "string",
            "seller_city": "string",
            "seller_state": "string",
        },
    },
    "product_category_name_translation.csv": {
        "dtype": {
            "product_category_name": "string",
            "product_category_name_english": "string",
        },
    },
}

# --- Utilidades ----------------------------------------------------
def normalize_text(s: pd.Series) -> pd.Series:
    """Quita espacios, pasa a minúsculas y elimina tildes/diacríticos."""
    s = s.astype("string").str.strip().str.lower()
    s = s.map(
        lambda x: (
            unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode()
            if pd.notna(x) else x
        )
    )
    return s

def timed(label: str):
    """Decorador para medir y reportar el tiempo de cada paso."""
    def deco(fn: Callable):
        def wrapper(*args, **kwargs):
            t0 = time.perf_counter()
            out = fn(*args, **kwargs)
            print(f"  ⏱  {label}: {time.perf_counter() - t0:.2f}s")
            return out
        return wrapper
    return deco

print(f"Proyecto             : {PROJECT_DIR}")
print(f"Datos crudos         : {RAW_DIR}")
print(f"Datos procesados     : {PROCESSED_DIR}")
print(f"Tabla analítica (mart): {MART_DIR}")


Proyecto             : C:\Users\User\Desktop\Materias ITM\Análisis de Datos\EntregaFinalAnalisisDatos
Datos crudos         : C:\Users\User\Desktop\Materias ITM\Análisis de Datos\EntregaFinalAnalisisDatos\Data
Datos procesados     : C:\Users\User\Desktop\Materias ITM\Análisis de Datos\EntregaFinalAnalisisDatos\Data\processed
Tabla analítica (mart): C:\Users\User\Desktop\Materias ITM\Análisis de Datos\EntregaFinalAnalisisDatos\Data\processed\marts


## 3. Extract — lectura tipada

Se cargan los 9 archivos con tipos explícitos y parseo de fechas. La
codificación se especifica para evitar el problema del **BOM** que trae
`product_category_name_translation.csv`.


In [2]:
@timed("Extract")
def extract_all() -> dict[str, pd.DataFrame]:
    """Carga los 9 CSV de Olist aplicando dtypes y parseo de fechas."""
    tables: dict[str, pd.DataFrame] = {}
    for fname, schema in SCHEMAS.items():
        path = RAW_DIR / fname
        df = pd.read_csv(
            path,
            dtype=schema.get("dtype"),
            parse_dates=schema.get("parse_dates"),
            encoding="utf-8-sig",   # tolera BOM
            low_memory=False,
        )
        key = fname.replace("olist_", "").replace("_dataset.csv", "").replace(".csv", "")
        tables[key] = df
    return tables

raw = extract_all()
for k, df in raw.items():
    print(f"{k:35s} -> {df.shape[0]:>7,} filas x {df.shape[1]:>2} cols   "
          f"({df.memory_usage(deep=True).sum()/1e6:5.1f} MB)")


  ⏱  Extract: 3.58s
customers                           ->  99,441 filas x  5 cols   ( 32.5 MB)
geolocation                         -> 1,000,163 filas x  5 cols   (181.7 MB)
orders                              ->  99,441 filas x  8 cols   ( 25.9 MB)
order_items                         -> 112,650 filas x  7 cols   ( 31.1 MB)
order_payments                      -> 103,886 filas x  5 cols   ( 17.2 MB)
order_reviews                       ->  99,224 filas x  7 cols   ( 32.7 MB)
products                            ->  32,951 filas x  9 cols   (  6.7 MB)
sellers                             ->   3,095 filas x  4 cols   (  0.8 MB)
product_category_name_translation   ->      71 filas x  2 cols   (  0.0 MB)


## 4. Profile — perfilado de calidad inicial

El perfilado nos da la línea base: cuántos nulos, duplicados y rangos
problemáticos hay en cada tabla. Estas métricas se vuelven a medir
después del *transform* para evidenciar la mejora.


In [3]:
def profile(df: pd.DataFrame, name: str) -> dict:
    """Devuelve métricas de calidad básicas de un DataFrame."""
    return {
        "tabla": name,
        "filas": len(df),
        "columnas": df.shape[1],
        "duplicados": int(df.duplicated().sum()),
        "celdas_nulas": int(df.isna().sum().sum()),
        "%_nulos": round(df.isna().sum().sum() / df.size * 100, 2),
        "memoria_mb": round(df.memory_usage(deep=True).sum() / 1e6, 2),
    }

profile_before = pd.DataFrame([profile(df, k) for k, df in raw.items()])
profile_before


,tabla,filas,columnas,duplicados,celdas_nulas,%_nulos,memoria_mb
0,customers,99441,5,0,0,0.00,32.45
1,geolocation,1000163,5,261831,0,0.00,182.60
2,orders,99441,8,0,4908,0.62,25.85
3,order_items,112650,7,0,0,0.00,31.09
4,order_payments,103886,5,0,0,0.00,17.23
5,order_reviews,99224,7,0,145903,21.01,32.72
6,products,32951,9,0,2448,0.83,6.72
7,sellers,3095,4,0,0,0.00,0.76
8,product_category_name_translation,71,2,0,0,0.00,0.01


In [4]:
# Detalle de nulos por columna en las tablas con mayor incidencia.
for k in ["orders", "order_reviews", "products"]:
    print(f"\n=== {k} — nulos por columna ===")
    nulls = raw[k].isna().sum()
    print(nulls[nulls > 0].to_string())



=== orders — nulos por columna ===
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965

=== order_reviews — nulos por columna ===
review_comment_title      87656
review_comment_message    58247

=== products — nulos por columna ===
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2


## 5. Transform — limpieza y construcción del modelo

Reglas aplicadas (documentadas por tabla):

| Tabla | Reglas |
|---|---|
| `customers` | normalización de `customer_city`, `customer_state`; chequeo de unicidad de `customer_id` |
| `sellers`   | normalización de ciudades/estados; unicidad de `seller_id` |
| `products`  | imputación de dimensiones físicas faltantes con la mediana de la categoría; *join* con la traducción de categorías |
| `geolocation` | colapso a un único registro por `zip_code_prefix` (mediana de lat/lng + ciudad/estado más frecuente) |
| `orders`    | dedup por `order_id`; cálculo de variables derivadas (`delivery_days_real`, `delivery_delay_days`, `is_late`) |
| `order_items` | filtrado por `order_id` válido; cálculo de `total_item_value = price + freight_value` |
| `payments`  | dedup por (`order_id`, `payment_sequential`) |
| `reviews`   | una review por orden (la última si hay duplicadas); flag `is_positive_review` (>= 4) |
| `dim_date`  | tabla generada a partir del rango de `order_purchase_timestamp` |


In [4]:
@timed("Transform")
def transform(raw: dict[str, pd.DataFrame]) -> dict[str, pd.DataFrame]:
    """Aplica las reglas de limpieza y devuelve dimensiones + hechos."""
    out: dict[str, pd.DataFrame] = {}

    # ---- dim_customers --------------------------------------------------
    cust = raw["customers"].drop_duplicates("customer_id").copy()
    cust["customer_city"]  = normalize_text(cust["customer_city"])
    cust["customer_state"] = cust["customer_state"].str.upper().str.strip()
    out["dim_customers"] = cust

    # ---- dim_sellers ----------------------------------------------------
    sell = raw["sellers"].drop_duplicates("seller_id").copy()
    sell["seller_city"]  = normalize_text(sell["seller_city"])
    sell["seller_state"] = sell["seller_state"].str.upper().str.strip()
    out["dim_sellers"] = sell

    # ---- dim_products + traducción de categorías -----------------------
    prod = raw["products"].drop_duplicates("product_id").copy()
    trans = raw["product_category_name_translation"]
    prod = prod.merge(trans, on="product_category_name", how="left")
    prod["product_category_name_english"] = prod["product_category_name_english"].fillna("unknown")

    # Imputación de dimensiones físicas (mediana por categoría)
    phys_cols = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]
    for c in phys_cols:
        med_cat = prod.groupby("product_category_name_english")[c].transform("median")
        prod[c] = prod[c].fillna(med_cat)
        prod[c] = prod[c].fillna(prod[c].median())
    out["dim_products"] = prod

    # ---- dim_geography (colapso por zip_code_prefix) -------------------
    # Para la moda (ciudad/estado más frecuente por zip) usamos
    # sort + drop_duplicates en lugar de groupby+lambda, mucho más rápido.
    geo = raw["geolocation"].copy()
    geo["geolocation_city"]  = normalize_text(geo["geolocation_city"])
    geo["geolocation_state"] = geo["geolocation_state"].str.upper().str.strip()
    geo_num = (geo.groupby("geolocation_zip_code_prefix", observed=True)
                  .agg(geolocation_lat=("geolocation_lat", "median"),
                       geolocation_lng=("geolocation_lng", "median"))
                  .reset_index())
    geo_city = (geo.groupby(["geolocation_zip_code_prefix","geolocation_city"], observed=True).size()
                  .reset_index(name="cnt")
                  .sort_values(["geolocation_zip_code_prefix","cnt"], ascending=[True, False])
                  .drop_duplicates("geolocation_zip_code_prefix")
                  [["geolocation_zip_code_prefix","geolocation_city"]])
    geo_state = (geo.groupby(["geolocation_zip_code_prefix","geolocation_state"], observed=True).size()
                   .reset_index(name="cnt")
                   .sort_values(["geolocation_zip_code_prefix","cnt"], ascending=[True, False])
                   .drop_duplicates("geolocation_zip_code_prefix")
                   [["geolocation_zip_code_prefix","geolocation_state"]])
    geo_agg = (geo_num.merge(geo_city,  on="geolocation_zip_code_prefix", how="left")
                       .merge(geo_state, on="geolocation_zip_code_prefix", how="left"))
    out["dim_geography"] = geo_agg

    # ---- fact_order_items + features de logística -----------------------
    orders = raw["orders"].drop_duplicates("order_id").copy()
    orders["delivery_days_real"] = (
        orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]
    ).dt.total_seconds() / 86400
    orders["delivery_days_estimated"] = (
        orders["order_estimated_delivery_date"] - orders["order_purchase_timestamp"]
    ).dt.total_seconds() / 86400
    orders["delivery_delay_days"] = (
        orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]
    ).dt.total_seconds() / 86400
    orders["is_late"] = (orders["delivery_delay_days"] > 0).astype("Int8")

    items = raw["order_items"].copy()
    items["total_item_value"] = items["price"] + items["freight_value"]

    # Solo ítems con orden existente
    items = items[items["order_id"].isin(orders["order_id"])]
    fact_items = items.merge(
        orders[[
            "order_id", "customer_id", "order_status",
            "order_purchase_timestamp", "order_delivered_customer_date",
            "delivery_days_real", "delivery_delay_days", "is_late",
        ]],
        on="order_id", how="left",
    )
    out["fact_order_items"] = fact_items

    # ---- fact_payments --------------------------------------------------
    pay = raw["order_payments"].drop_duplicates(["order_id", "payment_sequential"]).copy()
    pay = pay[pay["order_id"].isin(orders["order_id"])]
    out["fact_payments"] = pay

    # ---- dim_reviews (una por orden) -----------------------------------
    rev = raw["order_reviews"].copy()
    rev = (
        rev.sort_values("review_creation_date")
           .drop_duplicates("order_id", keep="last")
    )
    rev["is_positive_review"] = (rev["review_score"] >= 4).astype("Int8")
    out["dim_reviews"] = rev

    # ---- dim_date -------------------------------------------------------
    date_min = orders["order_purchase_timestamp"].min().normalize()
    date_max = orders["order_purchase_timestamp"].max().normalize()
    cal = pd.DataFrame({"date": pd.date_range(date_min, date_max, freq="D")})
    cal["year"]    = cal["date"].dt.year
    cal["quarter"] = cal["date"].dt.quarter
    cal["month"]   = cal["date"].dt.month
    cal["month_name"] = cal["date"].dt.strftime("%B")
    cal["week"]    = cal["date"].dt.isocalendar().week.astype("Int64")
    cal["day"]     = cal["date"].dt.day
    cal["day_of_week"] = cal["date"].dt.dayofweek
    cal["is_weekend"]  = cal["day_of_week"].isin([5, 6]).astype("Int8")
    out["dim_date"] = cal

    return out

clean = transform(raw)
for k, df in clean.items():
    print(f"{k:25s} -> {df.shape[0]:>7,} filas x {df.shape[1]:>2} cols")


  ⏱  Transform: 3.38s
dim_customers             ->  99,441 filas x  5 cols
dim_sellers               ->   3,095 filas x  4 cols
dim_products              ->  32,951 filas x 10 cols
dim_geography             ->  19,015 filas x  5 cols
fact_order_items          -> 112,650 filas x 15 cols
fact_payments             -> 103,886 filas x  5 cols
dim_reviews               ->  98,673 filas x  8 cols
dim_date                  ->     774 filas x  9 cols


## 6. Validate — integridad y reglas de negocio

Validaciones automatizadas. Cada chequeo emite **PASS** o **FAIL** con
contexto. Si alguna validación crítica falla, se levanta un
`AssertionError` para detener el pipeline.


In [5]:
class ValidationReport(list):
    def add(self, name: str, ok: bool, detail: str = ""):
        self.append({"check": name, "status": "PASS" if ok else "FAIL", "detail": detail})

    def to_frame(self) -> pd.DataFrame:
        return pd.DataFrame(self, columns=["check", "status", "detail"])

def validate(clean: dict[str, pd.DataFrame]) -> ValidationReport:
    rep = ValidationReport()

    cust = clean["dim_customers"]
    rep.add("PK dim_customers único",
            cust["customer_id"].is_unique,
            f"{cust['customer_id'].duplicated().sum()} duplicados")

    sell = clean["dim_sellers"]
    rep.add("PK dim_sellers único",
            sell["seller_id"].is_unique,
            f"{sell['seller_id'].duplicated().sum()} duplicados")

    prod = clean["dim_products"]
    rep.add("PK dim_products único",
            prod["product_id"].is_unique,
            f"{prod['product_id'].duplicated().sum()} duplicados")

    fact = clean["fact_order_items"]
    rep.add("FK customer_id en dim_customers",
            fact["customer_id"].isin(cust["customer_id"]).all(),
            f"{(~fact['customer_id'].isin(cust['customer_id'])).sum()} huérfanos")
    rep.add("FK seller_id en dim_sellers",
            fact["seller_id"].isin(sell["seller_id"]).all(),
            f"{(~fact['seller_id'].isin(sell['seller_id'])).sum()} huérfanos")
    rep.add("FK product_id en dim_products",
            fact["product_id"].isin(prod["product_id"]).all(),
            f"{(~fact['product_id'].isin(prod['product_id'])).sum()} huérfanos")

    pay = clean["fact_payments"]
    rep.add("PK fact_payments única (order_id+payment_sequential)",
            not pay.duplicated(["order_id", "payment_sequential"]).any(),
            f"{pay.duplicated(['order_id','payment_sequential']).sum()} duplicados")

    rep.add("price >= 0 en fact_order_items",
            (fact["price"] >= 0).all(),
            f"{(fact['price'] < 0).sum()} valores negativos")
    rep.add("freight_value >= 0 en fact_order_items",
            (fact["freight_value"] >= 0).all(),
            f"{(fact['freight_value'] < 0).sum()} valores negativos")
    rep.add("payment_value >= 0 en fact_payments",
            (pay["payment_value"] >= 0).all(),
            f"{(pay['payment_value'] < 0).sum()} valores negativos")

    rev = clean["dim_reviews"]
    rep.add("review_score en [1,5]",
            rev["review_score"].between(1, 5).all(),
            f"{(~rev['review_score'].between(1,5)).sum()} fuera de rango")
    rep.add("review por orden única",
            rev["order_id"].is_unique,
            f"{rev['order_id'].duplicated().sum()} duplicados")

    return rep

report = validate(clean).to_frame()
print(f"Validaciones: {(report['status']=='PASS').sum()}/{len(report)} PASS\n")
report


Validaciones: 12/12 PASS



,check,status,detail
0,PK dim_customers único,PASS,0 duplicados
1,PK dim_sellers único,PASS,0 duplicados
2,PK dim_products único,PASS,0 duplicados
3,FK customer_id en dim_customers,PASS,0 huérfanos
4,FK seller_id en dim_sellers,PASS,0 huérfanos
5,FK product_id en dim_products,PASS,0 huérfanos
6,PK fact_payments única (order_id+payment_seque...,PASS,0 duplicados
7,price >= 0 en fact_order_items,PASS,0 valores negativos
8,freight_value >= 0 en fact_order_items,PASS,0 valores negativos
9,payment_value >= 0 en fact_payments,PASS,0 valores negativos


In [7]:
# Bloqueo del pipeline si alguna validación crítica falló.
critical = report[report["status"] == "FAIL"]
assert critical.empty, f"Validaciones fallidas:\n{critical}"
print("✓ Todas las validaciones críticas pasaron.")


✓ Todas las validaciones críticas pasaron.


## 7. Load — persistencia de tablas limpias

Las dimensiones y los hechos se escriben en **CSV** dentro de
`Data/processed/`


In [6]:
@timed("Load tablas limpias (CSV)")
def load_clean(clean: dict[str, pd.DataFrame]) -> list[Path]:
    paths = []
    for name, df in clean.items():
        out = PROCESSED_DIR / f"{name}.csv"
        df.to_csv(out, index=False, encoding="utf-8")
        paths.append(out)
    return paths

clean_paths = load_clean(clean)
for p in clean_paths:
    print(f"{p.name:30s}  {p.stat().st_size/1e6:6.2f} MB")


  ⏱  Load tablas limpias (CSV): 3.12s
dim_customers.csv                 8.69 MB
dim_sellers.csv                   0.17 MB
dim_products.csv                  3.09 MB
dim_geography.csv                 1.14 MB
fact_order_items.csv             29.64 MB
fact_payments.csv                 5.74 MB
dim_reviews.csv                  13.51 MB
dim_date.csv                      0.03 MB


## 8. Tabla analítica (`mart_orders`)

Construimos una **tabla denormalizada por orden** (`mart_orders`)
pensada para ser consumida directamente por:

- el notebook de **EDA** (Fase 2),
- la herramienta de **BI** (Power BI, Fase 3),
- el **modelo predictivo** (Fase 4) — con `is_positive_review` como
  variable objetivo.

Tener una sola tabla amplia evita tener que volver a hacer joins en
cada fase y simplifica el trabajo en herramientas externas.


In [7]:
@timed("Build mart_orders")
def build_mart_orders(clean: dict[str, pd.DataFrame]) -> pd.DataFrame:
    fact = clean["fact_order_items"]
    pay  = clean["fact_payments"]
    rev  = clean["dim_reviews"]
    cust = clean["dim_customers"]
    prod = clean["dim_products"]
    sell = clean["dim_sellers"]

    # Agregaciones por orden a partir de los ítems
    items_per_order = (
        fact.groupby("order_id", observed=True)
            .agg(
                n_items=("order_item_id", "count"),
                n_distinct_products=("product_id", "nunique"),
                n_distinct_sellers=("seller_id", "nunique"),
                total_price=("price", "sum"),
                total_freight=("freight_value", "sum"),
                total_value=("total_item_value", "sum"),
                avg_item_price=("price", "mean"),
                first_seller_id=("seller_id", "first"),
                first_product_id=("product_id", "first"),
            )
            .reset_index()
    )

    # Pago consolidado por orden (numéricas vectoriales + moda con sort+drop_duplicates).
    pay_num = (pay.groupby("order_id", observed=True)
                  .agg(n_payments=("payment_sequential", "count"),
                       payment_total=("payment_value", "sum"),
                       payment_installments_max=("payment_installments", "max"))
                  .reset_index())
    pay_mode = (pay.groupby(["order_id","payment_type"], observed=True).size()
                  .reset_index(name="cnt")
                  .sort_values(["order_id","cnt"], ascending=[True, False])
                  .drop_duplicates("order_id")
                  [["order_id","payment_type"]]
                  .rename(columns={"payment_type":"main_payment_type"}))
    pay_per_order = pay_num.merge(pay_mode, on="order_id", how="left")

    # Cabecera de orden (un solo registro por order_id)
    head = (
        fact.drop_duplicates("order_id")[[
            "order_id", "customer_id",
            "order_status", "order_purchase_timestamp",
            "order_delivered_customer_date",
            "delivery_days_real", "delivery_delay_days", "is_late",
        ]]
    )

    mart = (
        head
        .merge(items_per_order, on="order_id", how="left")
        .merge(pay_per_order,   on="order_id", how="left")
        .merge(rev[["order_id", "review_score", "is_positive_review"]],
               on="order_id", how="left")
        .merge(cust[["customer_id", "customer_state", "customer_city"]],
               on="customer_id", how="left")
        .merge(sell[["seller_id", "seller_state"]]
               .rename(columns={"seller_id": "first_seller_id",
                                "seller_state": "first_seller_state"}),
               on="first_seller_id", how="left")
        .merge(prod[["product_id", "product_category_name_english"]]
               .rename(columns={"product_id": "first_product_id",
                                "product_category_name_english": "first_product_category"}),
               on="first_product_id", how="left")
    )

    # Variables temporales útiles para BI
    mart["purchase_year"]    = mart["order_purchase_timestamp"].dt.year
    mart["purchase_month"]   = mart["order_purchase_timestamp"].dt.to_period("M").astype(str)
    mart["purchase_dow"]     = mart["order_purchase_timestamp"].dt.dayofweek
    mart["is_weekend"]       = mart["purchase_dow"].isin([5, 6]).astype("Int8")

    return mart

mart_orders = build_mart_orders(clean)
print(f"mart_orders: {mart_orders.shape[0]:,} filas x {mart_orders.shape[1]} cols")
mart_orders.head()


  ⏱  Build mart_orders: 1.40s
mart_orders: 98,666 filas x 31 cols


,order_id,customer_id,order_status,order_purchase_timestamp,order_delivered_customer_date,delivery_days_real,delivery_delay_days,is_late,n_items,n_distinct_products,...,review_score,is_positive_review,customer_state,customer_city,first_seller_state,first_product_category,purchase_year,purchase_month,purchase_dow,is_weekend
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-20 23:43:48,7.614421,-8.011250,0,1,1,...,5,1,RJ,campos dos goytacazes,SP,cool_stuff,2017,2017-09,2,0
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-05-12 16:04:24,16.216181,-2.330278,0,1,1,...,4,1,SP,santa fe do sul,SP,pet_shop,2017,2017-04,2,0
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-22 13:19:16,7.948437,-13.444954,0,1,1,...,5,1,MG,para de minas,MG,furniture_decor,2018,2018-01,6,1
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-14 13:32:39,6.147269,-5.435660,0,1,1,...,4,1,SP,atibaia,SP,perfumery,2018,2018-08,2,0
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-03-01 16:42:31,25.114352,-15.303808,0,1,1,...,5,1,SP,varzea paulista,PR,garden_tools,2017,2017-02,5,1


In [8]:
@timed("Load mart_orders (CSV)")
def load_mart(mart: pd.DataFrame) -> Path:
    out = MART_DIR / "mart_orders.csv"
    mart.to_csv(out, index=False, encoding="utf-8")
    return out

mart_path = load_mart(mart_orders)
print(f"{mart_path.name:25s}  {mart_path.stat().st_size/1e6:6.2f} MB")


  ⏱  Load mart_orders (CSV): 2.02s
mart_orders.csv             32.24 MB


## 9. Reporte ejecutivo del ETL

Resumen final con métricas antes/después y trazabilidad por tabla.


In [9]:
profile_after = pd.DataFrame([
    profile(df, k) for k, df in clean.items()
])

print("=== Tablas limpias en disco ===")
print(pd.DataFrame({
    "archivo": [p.name for p in clean_paths] + [mart_path.name],
    "tamaño_MB": [round(p.stat().st_size / 1e6, 2)
                  for p in clean_paths + [mart_path]],
}).to_string(index=False))

print("\n=== Calidad después del ETL ===")
profile_after


=== Tablas limpias en disco ===
             archivo  tamaño_MB
   dim_customers.csv       8.69
     dim_sellers.csv       0.17
    dim_products.csv       3.09
   dim_geography.csv       1.14
fact_order_items.csv      29.64
   fact_payments.csv       5.74
     dim_reviews.csv      13.51
        dim_date.csv       0.03
     mart_orders.csv      32.24

=== Calidad después del ETL ===


,tabla,filas,columnas,duplicados,celdas_nulas,%_nulos,memoria_mb
0,dim_customers,99441,5,0,0,0.00,32.45
1,dim_sellers,3095,4,0,0,0.00,0.76
2,dim_products,32951,10,0,2440,0.74,8.75
3,dim_geography,19015,5,0,0,0.00,3.43
4,fact_order_items,112650,15,0,7362,0.44,51.48
5,fact_payments,103886,5,0,0,0.00,17.23
6,dim_reviews,98673,8,0,145018,18.37,33.53
7,dim_date,774,9,0,0,0.00,0.07


In [10]:
# Comparativa antes/después: % de nulos en las tablas relevantes.
before = profile_before.set_index("tabla")[["filas", "%_nulos"]]
before.columns = ["filas_before", "%_nulos_before"]
after  = profile_after.set_index("tabla")[["filas", "%_nulos"]]
after.columns  = ["filas_after",  "%_nulos_after"]

# Mapeo aproximado para alinear nombres
mapping = {
    "customers": "dim_customers",
    "sellers":   "dim_sellers",
    "products":  "dim_products",
    "geolocation": "dim_geography",
    "order_items": "fact_order_items",
    "order_payments": "fact_payments",
    "order_reviews": "dim_reviews",
}
cmp = before.rename(index=mapping).join(after, how="inner")
cmp["delta_filas"]  = cmp["filas_after"]  - cmp["filas_before"]
cmp["delta_%nulos"] = (cmp["%_nulos_after"] - cmp["%_nulos_before"]).round(2)
cmp


,filas_before,%_nulos_before,filas_after,%_nulos_after,delta_filas,delta_%nulos
tabla,,,,,,
dim_customers,99441,0.00,99441,0.00,0,0.00
dim_geography,1000163,0.00,19015,0.00,-981148,0.00
fact_order_items,112650,0.00,112650,0.44,0,0.44
fact_payments,103886,0.00,103886,0.00,0,0.00
dim_reviews,99224,21.01,98673,18.37,-551,-2.64
dim_products,32951,0.83,32951,0.74,0,-0.09
dim_sellers,3095,0.00,3095,0.00,0,0.00


## 10. Próximos pasos

1. **Fase 2 — Análisis Exploratorio de Datos (EDA):** se trabajará
   principalmente sobre `mart_orders.csv`, complementado con las
   dimensiones cuando sea necesario (categorías de producto, geografía).
2. **Fase 3 — Inteligencia de Negocios (BI):** consumir `mart_orders.csv`
   desde Power BI y construir el dashboard sobre el modelo
   estrella documentado en la sección 1.
3. **Fase 4 — Modelado predictivo:** entrenar un clasificador binario
   con target `is_positive_review` usando como features las variables
   logísticas (`delivery_days_real`, `is_late`), económicas
   (`payment_total`, `payment_installments_max`), de orden
   (`n_items`, `n_distinct_sellers`) y categóricas
   (`first_product_category`, `customer_state`, `main_payment_type`).
4. **Reproducibilidad:** este notebook está parametrizado por rutas
   relativas y funciones puras; basta con ejecutarlo de principio a fin
   para regenerar todos los CSV procesados.

---

**Archivos generados al ejecutar este notebook**

- `Data/processed/dim_customers.csv`
- `Data/processed/dim_sellers.csv`
- `Data/processed/dim_products.csv`
- `Data/processed/dim_geography.csv`
- `Data/processed/dim_date.csv`
- `Data/processed/dim_reviews.csv`
- `Data/processed/fact_order_items.csv`
- `Data/processed/fact_payments.csv`
- `Data/processed/marts/mart_orders.csv`
